In [7]:
import os 
from pathlib import Path
import hashlib
from PIL import Image
import pandas as pd
from collections import defaultdict
from sklearn.model_selection import train_test_split

In [2]:
os.getcwd()

'c:\\Users\\Lenovo\\Desktop\\project 2\\project2\\14_resNet'

# 1.1 — check classes

In [16]:
BASE_DIR = Path("newData")
SOURCE_FOLDERS = [
    BASE_DIR / "dataset" / 'train',
    BASE_DIR / "dataset" / 'clean',
    BASE_DIR / "datasetv2_TrainClean" / 'train',
    BASE_DIR / "datasetv2_TrainClean" / 'clean',
]
for p in SOURCE_FOLDERS:
    print(p.resolve(), "->", p.exists())

for folder in SOURCE_FOLDERS:
    print("\n", folder)
    if folder.exists():
        classes = [x.name for x in folder.iterdir() if x.is_dir()]
        print("Classes:", classes)
    else:
        print("Folder not found!")

C:\Users\Lenovo\Desktop\project 2\project2\14_resNet\newData\dataset\train -> True
C:\Users\Lenovo\Desktop\project 2\project2\14_resNet\newData\dataset\clean -> True
C:\Users\Lenovo\Desktop\project 2\project2\14_resNet\newData\datasetv2_TrainClean\train -> True
C:\Users\Lenovo\Desktop\project 2\project2\14_resNet\newData\datasetv2_TrainClean\clean -> True

 newData\dataset\train
Classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']

 newData\dataset\clean
Classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']

 newData\datasetv2_TrainClean\train
Classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']

 newData\datasetv2_TrainClean\clean
Classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']


# 1.2 — Checking format and corrupt file

In [17]:
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def build_inventory(source_folders):
    """
    از هر پوشه منبع زیرپوشه های کلاس را میخواند و یک جدول با
    filepath , label , source_folder میسازد
    """
    records = []
    for src in source_folders:
        if not src.exists():
            print(f"[WARNING] Path not found: {src}")
            continue
        for class_dir in sorted(src.iterdir()):
            if not class_dir.is_dir():
                continue
            label = class_dir.name
            for f in class_dir.rglob("*"):
                if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS:
                    records.append({
                        'filepath': str(f),
                        'label': label,
                        'source_folder': str(src),
                    })
    return pd.DataFrame(records)

# 1.3 — Duplicate Check

In [18]:
def compute_file_hash(filepath):
    """
    هش محتوای تصویر (نه فقط بایت های فایل) تا تصویر یکسان با فرمت/فشرده سازی
    متفاوت هم به عنوان duplicate شناسایی بشن
    """
    try:
        with Image.open(filepath) as img:
            img = img.convert("RGB").resize((64, 64))  # نرمال‌سازی قبل از هش
            return hashlib.md5(img.tobytes()).hexdigest()
    except Exception as e:
        return f"ERROR:{e}"

# 1.4 — Duplicate Finder

In [19]:
def find_duplicates(df):
    """
    هش هر عکس را حساب می‌کند و گروه‌های duplicate را برمی‌گرداند.
    """
    df = df.copy()
    df['content_hash'] = df['filepath'].apply(compute_file_hash)

    errors = df[df["content_hash"].str.startswith('ERROR', na=False)]
    if len(errors) > 0:
        print(f"[WARNING] {len(errors)}  Corrupt/unreadable file found:")
        print(errors[['filepath', 'content_hash']])

    valid_df = df[~df['content_hash'].str.startswith('ERROR', na=False)]
    dup_groups = valid_df.groupby("content_hash").filter(lambda g: len(g) > 1)
    return df, dup_groups

# 1.5 — Inventory Overview

In [ ]:
inventory_df = build_inventory(SOURCE_FOLDERS)

print("Total number of images:", len(inventory_df))
print("=" * 50)

print("\nNumber by class:")
print(inventory_df["label"].value_counts())

Total number of images: 3313

Number by class:
label
savari       498
kamyun       493
taxi         484
autobus      464
kamyunet     456
minibus      394
ambulance    369
vanet        155
Name: count, dtype: int64


In [25]:
print("Count by source and class:")
print(inventory_df.groupby(["source_folder", "label"]).size(), '\n')

Count by source and class:
source_folder                       label    
newData\dataset\clean               ambulance     49
                                    autobus       49
                                    kamyun        50
                                    kamyunet      49
                                    minibus       44
                                    savari        50
                                    taxi          50
                                    vanet         17
newData\dataset\train               ambulance     50
                                    autobus       48
                                    kamyun        50
                                    kamyunet      50
                                    minibus       49
                                    savari        50
                                    taxi          46
                                    vanet         15
newData\datasetv2_TrainClean\clean  ambulance    101
                          

# 1.6 — Duplicate Detection

In [27]:
print("Calculating hash to identify duplicates. this might take a while")
inventory_df, duplicates = find_duplicates(inventory_df)

print(f"\nNumber of duplicate groups found: {duplicates['content_hash'].nunique()}")
print(f"Number of duplicate images: {len(duplicates)}")

Calculating hash to identify duplicates. this might take a while

Number of duplicate groups found: 11
Number of duplicate images: 22


# 1.7 — Full Duplicate Report & Label Conflicts

In [29]:
print(f"All duplicate groups: {duplicates['content_hash'].nunique()}")
for h, group in duplicates.groupby("content_hash"):
    labels = group["label"].unique()
    conflict_flag = " <<< LABEL CONFLICT" if len(labels) > 1 else ""
    print(f"\nHash: {h}{conflict_flag}")
    print(group[["filepath", "label", "source_folder"]].to_string(index=False))

All duplicate groups: 11

Hash: 03a993ecd7c1867646e87b65c84d0998
                                     filepath     label         source_folder
newData\dataset\train\ambulance\213342200.jpg ambulance newData\dataset\train
newData\dataset\clean\ambulance\213342200.jpg ambulance newData\dataset\clean

Hash: 0e120336ec795b86a806f7ab6f4981e7
                                     filepath     label         source_folder
newData\dataset\train\ambulance\208738861.jpg ambulance newData\dataset\train
newData\dataset\clean\ambulance\208738861.jpg ambulance newData\dataset\clean

Hash: 2c40c39fa6d9123df3921e846153c3c8
                                     filepath     label         source_folder
newData\dataset\train\ambulance\213801906.jpg ambulance newData\dataset\train
newData\dataset\clean\ambulance\213801906.jpg ambulance newData\dataset\clean

Hash: 33e3f5d539b9ef54baf730105e0dbd16
                                   filepath   label         source_folder
newData\dataset\train\minibus\198331521

# 1.8 — Deduplication Function and Class Distribution After Deduplication

In [ ]:
def deduplicate(df, duplicates_df, prefer_source_keyword="clean"):
    """
    از هر گروه duplicate، نسخه‌ای که مسیرش شامل prefer_source_keyword است نگه داشته می‌شود؛
    بقیه از دیتافریم نهایی حذف می‌شوند (نه از دیسک).
    """
    drop_indices = []

    for content_hash, group in duplicates_df.groupby("content_hash"):
        preferred = group[group["filepath"].str.contains(prefer_source_keyword, case=False)]
        if len(preferred) == 0:
            # اگر هیچ‌کدام شامل کلمه‌ی ترجیحی نبود، فقط اولی را نگه دار
            keep_idx = group.index[0]
        else:
            keep_idx = preferred.index[0]

        drop_idx = [i for i in group.index if i != keep_idx]
        drop_indices.extend(drop_idx)

    cleaned_df = df.drop(index=drop_indices).reset_index(drop=True)
    print(f"Number of deleted rows:{len(drop_indices)}")
    print(f"Number of previous images: {len(df)} | After removing duplicates: {len(cleaned_df)}")
    return cleaned_df


inventory_clean_df = deduplicate(inventory_df, duplicates, prefer_source_keyword="clean")

print("\nFinal count by class (after removing duplicates):")
print(inventory_clean_df["label"].value_counts())

Number of deleted rows:11
Number of previous images: 3313 | After removing duplicates: 3302

Final count by class (after removing duplicates):
label
savari       498
kamyun       493
taxi         484
autobus      463
kamyunet     456
minibus      392
ambulance    361
vanet        155
Name: count, dtype: int64


# 1.9 — Defining Train and Validation Sets

In [32]:
SPLIT_SEED = 42
VAL_RATIO = 0.20

OUTPUT_DIR = BASE_DIR / "resnet_data_splits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def stratified_split(df, label_col="label", val_ratio=0.20, seed=42):
    """
    برای هر کلاس به‌طور جداگانه، val_ratio از نمونه‌ها به صورت تصادفی (با seed ثابت)
    به validation اختصاص داده می‌شود و بقیه train می‌شوند.
    """
    df = df.copy()
    df["split"] = ""

    for label, group in df.groupby(label_col):
        train_idx, val_idx = train_test_split(
            group.index,
            test_size=val_ratio,
            random_state=seed,
            shuffle=True,
        )
        df.loc[train_idx, "split"] = "train"
        df.loc[val_idx, "split"] = "val"

    return df

inventory_split_df = stratified_split(inventory_clean_df, label_col="label", val_ratio=VAL_RATIO,seed=SPLIT_SEED)

# ====== بررسی صحت split ======
print("Total number:", len(inventory_split_df))
print("\nTrain/val distribution by class:\n")
summary = inventory_split_df.groupby(["label", "split"]).size().unstack(fill_value=0)
summary["val_ratio_actual"] = (summary["val"] / (summary["train"] + summary["val"])).round(3)
print(summary)

Total number: 3302

Train/val distribution by class:

split      train  val  val_ratio_actual
label                                  
ambulance    288   73             0.202
autobus      370   93             0.201
kamyun       394   99             0.201
kamyunet     364   92             0.202
minibus      313   79             0.202
savari       398  100             0.201
taxi         387   97             0.200
vanet        124   31             0.200


# Checking for Data Leakage Between Train and Validation Sets

In [35]:
train_hashes = set(inventory_split_df.loc[inventory_split_df.split == "train", "content_hash"])
val_hashes = set(inventory_split_df.loc[inventory_split_df.split == "val", "content_hash"])
print("Hash overlap between train and val:", len(train_hashes & val_hashes))

Hash overlap between train and val: 0


# Save Split Manifest

In [33]:
output_path = OUTPUT_DIR / "inventory_train_val_split.csv"
inventory_split_df.to_csv(output_path, index=False)
print(f"\nThe split file was saved to this path:\n{output_path}")


The split file was saved to this path:
newData\resnet_data_splits\inventory_train_val_split.csv
